# Tropical Storm Bill: daily AIS traffic and monthly seaborne imports

This notebook links two different temporal levels around one recorded 2015 storm:

- **Daily AIS:** vessel crossings of the Galveston Bay entrance gate.
- **Monthly Census trade:** HS4 seaborne imports for Houston (5301), Texas City (5306), and Galveston (5310).

The event is Tropical Storm Bill, with Texas records in the NOAA source from June 15–17, 2015 and landfall on June 16. The two datasets are not forced into a false daily join: they are linked through a shared event and port-group definition, then plotted on their native frequencies. Missing AIS files remain missing rather than being interpreted as zero traffic.


In [ ]:
from pathlib import Path
from datetime import date, timedelta
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 30)

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root.')

PROJECT_ROOT = find_project_root()
PROJECT_ROOT


## 1. Reusable event and port-group configuration

To reuse this notebook for another storm or newly downloaded AIS year, change only `EVENT` and the window lengths. The file-discovery cell searches the configured year directories automatically.


In [ ]:
EVENT = {
    'name': 'Tropical Storm Bill',
    'episode_id': 97999,
    'event_start': date(2015, 6, 15),
    'event_date': date(2015, 6, 16),
    'event_end': date(2015, 6, 17),
    'state': 'TEXAS',
    'port_group_id': 'galveston_bay',
    'port_codes': ['5301', '5306', '5310'],
    'study_area': 'galveston_bay_entrance',
    'gate_file': 'galveston_bay_entrance.geojson',
    'landward_reference': (-94.95, 29.55),
}

AIS_PRE_DAYS = 21
AIS_POST_DAYS = 21
TRADE_PRE_MONTHS = 6
TRADE_POST_MONTHS = 6
TOP_HS4 = 5
MAX_TRACK_GAP_MINUTES = 60
MIN_SOG_KNOTS = 0.5
DEBOUNCE_HOURS = 6

PANEL_PATH = PROJECT_ROOT / 'data/interim/panel_port_hs4_month_2013_2025.parquet'
AIS_ROOT = PROJECT_ROOT / 'data/interim/ais_points_major_ports_2015_2024_v3'
GATE_PATH = PROJECT_ROOT / 'data/geofences/major_port_gates' / EVENT['gate_file']
NOAA_CANDIDATES = sorted((PROJECT_ROOT / 'data/raw/noaa').glob('noaa_storm_events_*combined*.parquet'))
NOAA_PATH = NOAA_CANDIDATES[-1] if NOAA_CANDIDATES else None
OUTPUT_DIR = PROJECT_ROOT / 'outputs/figures/explore'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert PANEL_PATH.exists(), PANEL_PATH
assert GATE_PATH.exists(), GATE_PATH
EVENT


## 2. Verify the event in the NOAA records

The panel's storm flag is repeated across HS4/country rows, so it must not be summed. The raw NOAA episode records are displayed here as event evidence.


In [ ]:
if NOAA_PATH is None:
    print('NOAA raw file not found; continuing with the configured event dates.')
    noaa_event = None
else:
    noaa_event = (
        pl.scan_parquet(NOAA_PATH)
        .filter(pl.col('episode_id') == EVENT['episode_id'])
        .select([
            'episode_id', 'event_id', 'state', 'event_type', 'cz_name',
            'begin_date_time', 'end_date_time', 'damage_property',
            'episode_narrative', 'event_narrative'
        ])
        .sort(['begin_date_time', 'event_id'])
        .collect()
    )
    display(noaa_event)


## 3. Discover available daily AIS files

The expected calendar is retained. Counts are set to zero only for dates whose Parquet file exists and contains no accepted crossings; dates without a file remain `NaN`.


In [ ]:
window_start = EVENT['event_date'] - timedelta(days=AIS_PRE_DAYS)
window_end = EVENT['event_date'] + timedelta(days=AIS_POST_DAYS)
expected_dates = pd.date_range(window_start, window_end, freq='D')

def ais_path_for_day(ts):
    d = ts.date()
    return AIS_ROOT / f'year={d.year}' / f'AIS_{d:%Y_%m_%d}.parquet'

availability = pd.DataFrame({'date': expected_dates})
availability['path'] = availability['date'].map(ais_path_for_day)
availability['ais_file_available'] = availability['path'].map(Path.exists)
available_paths = availability.loc[availability['ais_file_available'], 'path'].tolist()

print(f"AIS window: {window_start} to {window_end}")
print(f"Available files: {len(available_paths)} / {len(availability)}")
display(availability.loc[~availability['ais_file_available'], ['date', 'path']])
if not available_paths:
    raise FileNotFoundError('No AIS files are available in the configured event window.')


## 4. Detect finite-line gate crossings

A crossing is accepted when consecutive observations for the same MMSI cross the finite gate segment, are no more than 60 minutes apart, have SOG at least 0.5 knots, and are at least six hours after that vessel's previous accepted crossing. Direction is defined relative to a landward reference point. These rules match the gate-crossing logic used in the AIS exploration workflow.


In [ ]:
with GATE_PATH.open() as f:
    gate_geojson = json.load(f)
gate_coordinates = gate_geojson['features'][0]['geometry']['coordinates']
gate_a = tuple(gate_coordinates[0])
gate_b = tuple(gate_coordinates[-1])

ais = (
    pl.scan_parquet([str(p) for p in available_paths])
    .filter(pl.col('study_area') == EVENT['study_area'])
    .select(['MMSI', 'BaseDateTime', 'LAT', 'LON', 'SOG', 'vessel_group', 'study_area'])
    .collect()
    .to_pandas()
)
ais['BaseDateTime'] = pd.to_datetime(ais['BaseDateTime'], utc=True)
ais = ais.sort_values(['MMSI', 'BaseDateTime']).reset_index(drop=True)
print(f"Loaded {len(ais):,} AIS points for {EVENT['study_area']}.")

def orientation(ax, ay, bx, by, px, py):
    return (bx - ax) * (py - ay) - (by - ay) * (px - ax)

def detect_gate_crossings(points, gate_a, gate_b, landward_reference):
    df = points.copy()
    grouped = df.groupby('MMSI', sort=False)
    df['prev_time'] = grouped['BaseDateTime'].shift()
    df['prev_lat'] = grouped['LAT'].shift()
    df['prev_lon'] = grouped['LON'].shift()
    df['gap_minutes'] = (df['BaseDateTime'] - df['prev_time']).dt.total_seconds() / 60

    ax, ay = gate_a
    bx, by = gate_b
    df['side_prev'] = orientation(ax, ay, bx, by, df['prev_lon'], df['prev_lat'])
    df['side_curr'] = orientation(ax, ay, bx, by, df['LON'], df['LAT'])
    gate_side_prev = orientation(df['prev_lon'], df['prev_lat'], df['LON'], df['LAT'], ax, ay)
    gate_side_curr = orientation(df['prev_lon'], df['prev_lat'], df['LON'], df['LAT'], bx, by)

    crosses_infinite_line = (df['side_prev'] * df['side_curr']) < 0
    intersects_finite_gate = (gate_side_prev * gate_side_curr) <= 0
    valid_track = df['gap_minutes'].between(0, MAX_TRACK_GAP_MINUTES, inclusive='both')
    moving = df['SOG'].fillna(0) >= MIN_SOG_KNOTS
    candidates = df.loc[crosses_infinite_line & intersects_finite_gate & valid_track & moving].copy()

    landward_side = orientation(ax, ay, bx, by, *landward_reference)
    candidates['direction'] = np.where(
        np.sign(candidates['side_curr']) == np.sign(landward_side), 'inbound', 'outbound'
    )
    candidates = candidates.sort_values(['BaseDateTime', 'MMSI'])

    accepted = []
    last_crossing = {}
    debounce = pd.Timedelta(hours=DEBOUNCE_HOURS)
    for idx, row in candidates.iterrows():
        previous = last_crossing.get(row['MMSI'])
        if previous is None or row['BaseDateTime'] - previous >= debounce:
            accepted.append(idx)
            last_crossing[row['MMSI']] = row['BaseDateTime']
    return candidates.loc[accepted].copy()

crossings = detect_gate_crossings(ais, gate_a, gate_b, EVENT['landward_reference'])
crossings['date'] = crossings['BaseDateTime'].dt.tz_convert(None).dt.normalize()
print(f"Accepted gate crossings: {len(crossings):,}")
display(crossings.head())


In [ ]:
daily_counts = (
    crossings.groupby(['date', 'direction']).size()
    .unstack(fill_value=0)
    .reindex(columns=['inbound', 'outbound'], fill_value=0)
    .reset_index()
)
daily = availability[['date', 'ais_file_available']].merge(daily_counts, on='date', how='left')
for col in ['inbound', 'outbound']:
    daily.loc[daily['ais_file_available'] & daily[col].isna(), col] = 0
daily['total'] = daily[['inbound', 'outbound']].sum(axis=1, min_count=2)
daily['total_7d_mean'] = daily['total'].rolling(7, center=True, min_periods=4).mean()
daily['relative_day'] = (daily['date'] - pd.Timestamp(EVENT['event_date'])).dt.days

baseline_mask = (daily['date'] < pd.Timestamp(EVENT['event_start'])) & daily['ais_file_available']
event_mask = daily['date'].between(pd.Timestamp(EVENT['event_start']), pd.Timestamp(EVENT['event_end'])) & daily['ais_file_available']
baseline_mean = daily.loc[baseline_mask, 'total'].mean()
event_mean = daily.loc[event_mask, 'total'].mean()
ais_change_pct = 100 * (event_mean / baseline_mean - 1)

ais_summary = pd.DataFrame({
    'metric': ['baseline daily mean', 'event-window daily mean', 'event vs baseline'],
    'value': [baseline_mean, event_mean, ais_change_pct],
    'unit': ['crossings/day', 'crossings/day', 'percent'],
})
display(ais_summary.round(2))
display(daily.loc[daily['date'].between('2015-06-13', '2015-06-19')])


## 5. Build the monthly HS4 import view

`CTY_CODE == '-'` selects the Census port-total rows and prevents double-counting country detail. `COMM_LVL == 'HS4'` fixes the commodity level. `VES_VAL_MO` is used instead of all-mode imports because the comparison is with vessel traffic. Top commodities are selected using only pre-event months, avoiding post-treatment selection.


In [ ]:
event_period = pd.Period(EVENT['event_date'], freq='M')
trade_start = event_period - TRADE_PRE_MONTHS
trade_end = event_period + TRADE_POST_MONTHS

trade = (
    pl.scan_parquet(PANEL_PATH)
    .filter(pl.col('PORT').cast(pl.Utf8).is_in(EVENT['port_codes']))
    .filter(pl.col('CTY_CODE').cast(pl.Utf8) == '-')
    .filter(pl.col('COMM_LVL') == 'HS4')
    .filter(
        (pl.col('time') >= pl.lit(str(trade_start)))
        & (pl.col('time') <= pl.lit(str(trade_end)))
    )
    .select([
        'time', 'PORT', 'PORT_NAME', 'I_COMMODITY', 'I_COMMODITY_LDESC',
        'VES_VAL_MO', 'VES_WGT_MO', 'n_storm_events', 'storm_month'
    ])
    .collect()
    .to_pandas()
)
if trade.empty:
    raise ValueError('No port-total HS4 rows found for the configured ports and months.')
trade['month'] = pd.PeriodIndex(trade['time'], freq='M')
trade['VES_VAL_MO'] = pd.to_numeric(trade['VES_VAL_MO'], errors='coerce').fillna(0)

pre_event_trade = trade[trade['month'] < event_period]
top_hs4 = (
    pre_event_trade.groupby(['I_COMMODITY', 'I_COMMODITY_LDESC'], as_index=False)['VES_VAL_MO'].sum()
    .nlargest(TOP_HS4, 'VES_VAL_MO')
)
top_codes = top_hs4['I_COMMODITY'].tolist()
label_map = {
    row.I_COMMODITY: f"{row.I_COMMODITY} {row.I_COMMODITY_LDESC[:38].title()}"
    for row in top_hs4.itertuples()
}

all_months = pd.period_range(trade_start, trade_end, freq='M')
total_monthly = trade.groupby('month')['VES_VAL_MO'].sum().reindex(all_months, fill_value=0)
hs4_monthly = (
    trade[trade['I_COMMODITY'].isin(top_codes)]
    .pivot_table(index='month', columns='I_COMMODITY', values='VES_VAL_MO', aggfunc='sum', fill_value=0)
    .reindex(all_months, fill_value=0)
)
hs4_monthly = hs4_monthly.rename(columns=label_map)
pre_period_mask = hs4_monthly.index < event_period
hs4_index = hs4_monthly.divide(hs4_monthly.loc[pre_period_mask].mean()).multiply(100)
trade_plot_dates = all_months.to_timestamp(how='start')

trade_summary = pd.DataFrame({
    'month': all_months.astype(str),
    'vessel_import_value_usd': total_monthly.to_numpy(),
    'relative_month': np.arange(-TRADE_PRE_MONTHS, TRADE_POST_MONTHS + 1),
})
display(top_hs4)
display(trade_summary)


## 6. Linked event-window plots

The shaded region is the NOAA event window. The monthly trade panels cannot identify the exact day of disruption; they show whether June and the following months are unusual relative to the preceding six months. Commodity lines are normalized to a pre-event mean of 100 so differently sized HS4 categories can be compared.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 14), constrained_layout=True)
event_start_ts = pd.Timestamp(EVENT['event_start'])
event_end_ts = pd.Timestamp(EVENT['event_end']) + pd.Timedelta(days=1)

ax = axes[0]
ax.plot(daily['date'], daily['inbound'], marker='o', ms=3, lw=1.2, label='Inbound')
ax.plot(daily['date'], daily['outbound'], marker='o', ms=3, lw=1.2, label='Outbound')
ax.plot(daily['date'], daily['total_7d_mean'], color='black', lw=2.2, label='Total, centered 7-day mean')
ax.axvspan(event_start_ts, event_end_ts, color='tab:red', alpha=0.18, label='NOAA event window')
missing = daily.loc[~daily['ais_file_available'], 'date']
for d in missing:
    ax.axvspan(d, d + pd.Timedelta(days=1), color='0.75', alpha=0.5)
ax.set(title=f"{EVENT['name']}: Galveston Bay entrance crossings", ylabel='Accepted vessel crossings/day')
ax.legend(ncol=2)

ax = axes[1]
ax.plot(trade_plot_dates, total_monthly.to_numpy() / 1e9, marker='o', lw=2, color='tab:blue')
ax.axvspan(pd.Timestamp(event_period.start_time), pd.Timestamp(event_period.end_time), color='tab:red', alpha=0.18)
ax.set(title='Monthly seaborne import value: Houston + Texas City + Galveston', ylabel='VES_VAL_MO (USD billions)')

ax = axes[2]
for col in hs4_index.columns:
    ax.plot(trade_plot_dates, hs4_index[col], marker='o', lw=1.5, label=col)
ax.axhline(100, color='black', lw=1, ls='--')
ax.axvspan(pd.Timestamp(event_period.start_time), pd.Timestamp(event_period.end_time), color='tab:red', alpha=0.18)
ax.set(title=f'Top {TOP_HS4} pre-event HS4 imports (pre-event mean = 100)', ylabel='Import value index', xlabel='Date')
ax.legend(fontsize=8, ncol=2)

figure_path = OUTPUT_DIR / 'tropical_storm_bill_ais_trade_event_window.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
figure_path


## 7. Compact linked summary and interpretation guardrails

A sharp daily AIS decline during the event is direct descriptive evidence of a short port-access disruption. A June monthly import change is much coarser: cargo may arrive before or after the storm within the same month, and oil prices/composition can move value independently of vessel counts. Treat this plot as case-selection and validation evidence, not a causal estimate. The wider 2013–2025 panel should be used for formal treated-versus-control analysis and rerouting tests.


In [ ]:
pre_trade_mean = total_monthly.loc[total_monthly.index < event_period].mean()
event_trade_value = total_monthly.loc[event_period]
trade_change_pct = 100 * (event_trade_value / pre_trade_mean - 1)

linked_summary = pd.DataFrame([{
    'event': EVENT['name'],
    'event_date': EVENT['event_date'],
    'port_group_id': EVENT['port_group_id'],
    'port_codes': ','.join(EVENT['port_codes']),
    'ais_baseline_crossings_per_day': baseline_mean,
    'ais_event_crossings_per_day': event_mean,
    'ais_event_change_pct': ais_change_pct,
    'pre_event_monthly_import_mean_usd': pre_trade_mean,
    'event_month_import_usd': event_trade_value,
    'event_month_import_change_pct': trade_change_pct,
    'ais_files_available': int(availability['ais_file_available'].sum()),
    'ais_files_expected': len(availability),
}])
display(linked_summary.round(2))
print(f"AIS event-window change: {ais_change_pct:.1f}% relative to the available pre-event days.")
print(f"June 2015 vessel-import value change: {trade_change_pct:.1f}% relative to the preceding six-month mean.")
